# 🚀 Install ALL Services (Cassandra + Milvus + Redis + Neo4j)

This notebook installs the ACTUAL services, not mocks!

In [ ]:
# Setup repository
import os

if os.path.exists("full_architecture_chat.py"):
    pass
else:
    !git clone https://github.com/champi-dev/Think_AI .

In [ ]:
# Fix marshmallow first
!pip install marshmallow==3.20.1 --force-reinstall

In [ ]:
# 1. Install and start Redis
!apt-get update -qq && apt-get install -y redis-server
!redis-server --daemonize yes
!redis-cli ping

In [ ]:
# 2. Neo4j is already running (from your output)
!nc -zv localhost 7687 && echo "✅ Neo4j is already running!"

In [ ]:
# 3. Install Cassandra (WORKING VERSION)
%%bash
# Install Java
apt-get install -y openjdk-11-jre-headless

# Download Cassandra 4.0.11 (stable version)
cd /opt
wget -q https://archive.apache.org/dist/cassandra/4.0.11/apache-cassandra-4.0.11-bin.tar.gz
tar -xzf apache-cassandra-4.0.11-bin.tar.gz
mv apache-cassandra-4.0.11 cassandra

# Configure for Colab (limited memory)
echo "JVM_OPTS=\"\$JVM_OPTS -Xms256M\"" >> /opt/cassandra/conf/cassandra-env.sh
echo "JVM_OPTS=\"\$JVM_OPTS -Xmx512M\"" >> /opt/cassandra/conf/cassandra-env.sh

# Start Cassandra
/opt/cassandra/bin/cassandra -R &

echo "Cassandra starting..."

In [ ]:
# 4. Install Milvus using Python server
!pip install milvus-server

# Start Milvus server in background
import subprocess

milvus_proc = subprocess.Popen(
    ["python", "-c", "from milvus import default_server; default_server.start()"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)


In [ ]:
# Wait for all services to start
import time

time.sleep(30)

# Check all services
!echo "\n🔍 Checking all services:"
!nc -zv localhost 6379 2>&1 | grep succeeded && echo "✅ Redis: RUNNING" || echo "❌ Redis: NOT READY"
!nc -zv localhost 7687 2>&1 | grep succeeded && echo "✅ Neo4j: RUNNING" || echo "❌ Neo4j: NOT READY"
!nc -zv localhost 9042 2>&1 | grep succeeded && echo "✅ Cassandra: RUNNING" || echo "❌ Cassandra: NOT READY"
!nc -zv localhost 19530 2>&1 | grep succeeded && echo "✅ Milvus: RUNNING" || echo "❌ Milvus: NOT READY"

In [ ]:
# If Cassandra isn't responding, check logs
!tail -20 /opt/cassandra/logs/system.log 2>/dev/null || echo "No Cassandra logs yet"

In [ ]:
# Alternative: Use Cassandra Python driver with embedded server
!pip install cassandra-driver scylla-driver

# Test connection
from cassandra.cluster import Cluster

try:
    cluster = Cluster(["127.0.0.1"])
    session = cluster.connect()
    cluster.shutdown()
except Exception:

    # Use in-memory alternative
    !pip install astrapy

In [ ]:
# Install Python dependencies
!pip install -r requirements.txt

In [ ]:
# Run the FULL system!
!python full_architecture_chat.py